# Real-Time Inference Simulation — Vishing Detection

Runs the inference simulator over the 100K synthetic sessions by sending each observation
**one by one** to the `VishingModelWrapper`, measures the latency in ms, and generates the results dashboard.

**Prerequisites:**
- `10_Inference_Data_Generation.ipynb` executed → parquet in S3
- At least one trained model (Notebook 7 or Notebook 9) available in S3
- `realtime_simulator.py` in the same working directory

## Configuration (the only cell to edit)

In [ ]:
# ── CONFIGURE HERE ────────────────────────────────────────────────────────────

# S3 path of the VishingModelWrapper to evaluate
WRAPPER_S3_PATH = 's3://poc-fraude-vishing/proyecto/modelos_xgb/augmented/xgb_deep/random_oversampling/25.pkl'

# S3 path of the inference dataset generated in Notebook 10
DATA_S3_PATH = 's3://poc-fraude-vishing/proyecto/data/inference_simulation/inference_100k.parquet'

BUCKET         = 'poc-fraude-vishing'
RESULTS_PREFIX = 'proyecto/data/inference_simulation/results'

# ─────────────────────────────────────────────────────────────────────────────
print(f'Model : {WRAPPER_S3_PATH}')
print(f'Data  : {DATA_S3_PATH}')

In [ ]:
%pip install --quiet "sagemaker<3" tqdm

In [ ]:
import pandas as pd
import numpy as np
import boto3
import sagemaker
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from realtime_simulator import RealtimeInferenceSimulator

sagemaker_session = sagemaker.Session()
print('Libraries loaded.')

In [ ]:
import realtime_simulator

print(dir(realtime_simulator))

In [ ]:
class VishingModelWrapper:
    """
    Bundles a trained XGBoost model with its feature list and optimal
    decision threshold. Accepts dict, JSON string, or list[dict] as input.
    """

    def __init__(self, model, feature_names, threshold=0.5,
                 model_name='', technique='', ratio='', data_type=''):
        self.model         = model
        self.feature_names = list(feature_names)
        self.threshold     = threshold
        self.model_name    = model_name
        self.technique     = technique
        self.ratio         = ratio
        self.data_type     = data_type  # 'augmented' | 'original'

    def _to_array(self, json_input):
        if isinstance(json_input, str):
            data = _json.loads(json_input)
        elif isinstance(json_input, dict):
            data = json_input
        elif isinstance(json_input, list):
            return np.vstack([self._to_array(item) for item in json_input])
        else:
            raise TypeError(f'Expected dict, JSON string, or list. Got {type(json_input)}')
        missing = set(self.feature_names) - set(data.keys())
        if missing:
            raise ValueError(f'Missing features: {sorted(missing)}')
        return np.array([[data[f] for f in self.feature_names]], dtype=np.float64)

    def predict(self, json_input):
        proba  = self.model.predict_proba(self._to_array(json_input))[:, 1]
        labels = (proba >= self.threshold).astype(int).tolist()
        return labels[0] if len(labels) == 1 else labels

    def predict_proba_raw(self, json_input):
        proba = self.model.predict_proba(self._to_array(json_input))
        rows  = [{'legitimate': round(float(p[0]), 6), 'vishing': round(float(p[1]), 6)}
                 for p in proba]
        return rows[0] if len(rows) == 1 else rows

    def predict_full(self, json_input):
        proba   = self.model.predict_proba(self._to_array(json_input))
        results = []
        for p in proba:
            label = int(p[1] >= self.threshold)
            results.append({
                'prediction':             label,
                'label':                  'vishing' if label == 1 else 'legitimate',
                'probability_vishing':    round(float(p[1]), 6),
                'probability_legitimate': round(float(p[0]), 6),
                'threshold_used':         round(self.threshold, 6),
            })
        return results[0] if len(results) == 1 else results

    def __repr__(self):
        return (f'VishingModelWrapper(variant={self.model_name!r}, '
                f'data={self.data_type!r}, technique={self.technique!r}, '
                f'ratio={self.ratio!r}, n_features={len(self.feature_names)}, '
                f'threshold={self.threshold:.4f})')

## 1. Simulator creation

In [ ]:
simulator = RealtimeInferenceSimulator(
    wrapper_s3_path   = WRAPPER_S3_PATH,
    data_s3_path      = DATA_S3_PATH,
    bucket            = BUCKET,
    results_s3_prefix = RESULTS_PREFIX,
)

# Explicit loading to inspect before running
simulator.load_model()
simulator.load_data()

## 2. Running the real-time flow
Each observation is sent individually as a `dict` to the wrapper and the latency is measured with `time.perf_counter()`.

In [ ]:
df_results = simulator.run(show_progress=True)

## 3. Save results to S3 and compute metrics

In [ ]:
results_path = simulator.save_results('simulation_results.csv')

metrics = simulator.compute_metrics()

print('\n' + '='*65)
print('  SIMULATION SUMMARY')
print('='*65)
print(f'  Observations processed   : {metrics["total_obs"]:>10,}')

if 'total_vishing' in metrics:
    print(f'  Actual vishing sessions  : {metrics["total_vishing"]:>10,}  '
          f'({metrics["total_vishing"]/metrics["total_obs"]*100:.2f}%)')
    print(f'  Actual legitimate sessions: {metrics["total_legitimate"]:>10,}')

print(f'\n  ALERTS GENERATED')
print(f'  Total alerts             : {metrics["total_alerts"]:>10,}')
print(f'  Alert rate               : {metrics["alert_rate_pct"]:>10.4f}%')

if 'recall' in metrics:
    print(f'\n  DETECTION METRICS')
    print(f'  Vishing detected (TP)    : {metrics["vishing_detected"]:>10,}')
    print(f'  Vishing missed (FN)      : {metrics["vishing_missed"]:>10,}')
    print(f'  False alerts (FP)        : {metrics["false_alerts"]:>10,}')
    print(f'  Recall                   : {metrics["recall"]:>10.4f}')
    print(f'  Precision                : {metrics["precision"]:>10.4f}')
    print(f'  F1                       : {metrics["f1"]:>10.4f}')

print(f'\n  INFERENCE LATENCY (ms)')
print(f'  Mean                     : {metrics["lat_mean_ms"]:>10.4f} ms')
print(f'  Median (p50)             : {metrics["lat_median_ms"]:>10.4f} ms')
print(f'  p95                      : {metrics["lat_p95_ms"]:>10.4f} ms')
print(f'  p99                      : {metrics["lat_p99_ms"]:>10.4f} ms')
print(f'  Maximum                  : {metrics["lat_max_ms"]:>10.4f} ms')
print('='*65)

## 4. Results dashboard
### 4.1 Latency distribution

In [ ]:
lat = df_results['latency_ms']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(lat, bins=80, color='#3498db', edgecolor='white', linewidth=0.3)
axes[0].axvline(metrics['lat_median_ms'], color='orange', linestyle='--',
                linewidth=2, label=f'p50 = {metrics["lat_median_ms"]:.3f} ms')
axes[0].axvline(metrics['lat_p95_ms'],    color='red',    linestyle='--',
                linewidth=2, label=f'p95 = {metrics["lat_p95_ms"]:.3f} ms')
axes[0].axvline(metrics['lat_p99_ms'],    color='darkred', linestyle=':',
                linewidth=2, label=f'p99 = {metrics["lat_p99_ms"]:.3f} ms')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Latency Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# Boxplot by predicted class
df_results['Prediction'] = df_results['label_pred'].map(
    {'legitimate': 'Legitimate', 'vishing': 'Vishing'})
bp_data = [df_results[df_results['label_pred'] == c]['latency_ms'].values
           for c in ['legitimate', 'vishing']]
axes[1].boxplot(bp_data, labels=['Legitimate', 'Vishing'],
                patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.6),
                medianprops=dict(color='orange', linewidth=2))
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency by Predicted Class', fontweight='bold')

# Cumulative percentiles
pcts  = np.arange(1, 100)
vals  = np.percentile(lat, pcts)
axes[2].plot(pcts, vals, color='#3498db', linewidth=2)
axes[2].axhline(metrics['lat_p95_ms'], color='red',    linestyle='--',
                alpha=0.7, label=f'p95 = {metrics["lat_p95_ms"]:.3f} ms')
axes[2].axhline(metrics['lat_p99_ms'], color='darkred', linestyle=':',
                alpha=0.7, label=f'p99 = {metrics["lat_p99_ms"]:.3f} ms')
axes[2].set_xlabel('Percentile')
axes[2].set_ylabel('Latency (ms)')
axes[2].set_title('Latency Percentile Curve', fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Latency Analysis — Real-Time Inference (100K obs.)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Score distribution and detection

In [ ]:
has_gt = 'is_vishing_real' in df_results.columns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# score_vishing distribution
if has_gt:
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df_results[df_results['is_vishing_real'] == label]['score_vishing']
        axes[0].hist(subset, bins=60, alpha=0.6, color=color,
                     label=f'{"Legitimate" if label==0 else "Vishing"} (n={len(subset):,})',
                     density=True)
else:
    axes[0].hist(df_results['score_vishing'], bins=60, color='#3498db', alpha=0.7, density=True)

thr = df_results['threshold_used'].iloc[0]
axes[0].axvline(thr, color='black', linestyle='--', linewidth=2, label=f'Threshold = {thr:.4f}')
axes[0].set_xlabel('Vishing score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# Confusion matrix (if ground truth available)
if has_gt:
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(df_results['is_vishing_real'], df_results['prediction'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['Legitimate (pred)', 'Vishing (pred)'],
                yticklabels=['Legitimate (actual)', 'Vishing (actual)'])
    axes[1].set_title('Confusion Matrix', fontweight='bold')

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[2],
                xticklabels=['Legitimate (pred)', 'Vishing (pred)'],
                yticklabels=['Legitimate (actual)', 'Vishing (actual)'])
    axes[2].set_title('Confusion Matrix — Normalized', fontweight='bold')
else:
    counts = df_results['label_pred'].value_counts()
    axes[1].bar(counts.index, counts.values,
                color=['#2ecc71', '#e74c3c'])
    axes[1].set_title('Prediction Distribution', fontweight='bold')
    axes[2].axis('off')

plt.suptitle('Detection Analysis — Real-Time Simulation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Latency over time (first 5,000 observations)

In [ ]:
sample = df_results.head(5_000).copy()
sample['rolling_p95'] = sample['latency_ms'].rolling(500, min_periods=1).quantile(0.95)

fig, ax = plt.subplots(figsize=(16, 4))

if has_gt:
    colors = sample['is_vishing_real'].map({0: '#3498db', 1: '#e74c3c'})
    ax.scatter(sample.index, sample['latency_ms'], c=colors, s=3, alpha=0.4)
    ax.plot([], [], 'o', color='#3498db', markersize=5, label='Legitimate')
    ax.plot([], [], 'o', color='#e74c3c', markersize=5, label='Vishing')
else:
    ax.scatter(sample.index, sample['latency_ms'], s=3, alpha=0.3, color='#3498db')

ax.plot(sample.index, sample['rolling_p95'], color='orange', linewidth=1.5,
        label='rolling p95 (window=500)')
ax.axhline(metrics['lat_p95_ms'], color='red', linestyle='--', alpha=0.6,
           label=f'global p95 = {metrics["lat_p95_ms"]:.3f} ms')
ax.set_xlabel('Observation number')
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency per Observation — First 5,000', fontweight='bold')
ax.legend(fontsize=9, markerscale=3)

plt.tight_layout()
plt.show()

### 4.4 Risk score over time (timeline)

In [ ]:
sample_tl = df_results.head(2_000).copy()
sample_tl['rolling_score'] = sample_tl['score_vishing'].rolling(100, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(16, 4))

ax.fill_between(sample_tl.index, sample_tl['rolling_score'],
                alpha=0.3, color='#e74c3c', label='Vishing score (moving average)')
ax.plot(sample_tl.index, sample_tl['rolling_score'], color='#e74c3c', linewidth=1)
ax.axhline(thr, color='black', linestyle='--', linewidth=1.5,
           label=f'Decision threshold = {thr:.4f}')

if has_gt:
    vishing_idx = sample_tl[sample_tl['is_vishing_real'] == 1].index
    ax.scatter(vishing_idx, [thr * 0.95] * len(vishing_idx),
               marker='|', color='red', s=80, linewidths=1.5, label='Actual vishing session')

ax.set_xlabel('Observation number')
ax.set_ylabel('Vishing score (moving average, window=100)')
ax.set_title('Risk Timeline — Real-Time Simulation', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, None)

plt.tight_layout()
plt.show()

## 5. Sample view of results

In [ ]:
display_cols = ['obs_index', 'score_vishing', 'score_legitimate',
                'label_pred', 'threshold_used', 'latency_ms']
if has_gt:
    display_cols.insert(2, 'is_vishing_real')

# Sample: 5 alerts + 5 legitimate
alerts   = df_results[df_results['prediction'] == 1].head(5)
legit    = df_results[df_results['prediction'] == 0].head(5)
sample_display = pd.concat([alerts, legit])

print('Sample results (5 alerts + 5 legitimate):')
display(
    sample_display[display_cols]
    .style
    .background_gradient(cmap='RdYlGn_r', subset=['score_vishing'])
    .background_gradient(cmap='Blues', subset=['latency_ms'])
    .format({'score_vishing': '{:.6f}', 'score_legitimate': '{:.6f}',
             'latency_ms': '{:.4f} ms', 'threshold_used': '{:.6f}'})
)

print(f'\nFull results available at: {results_path}')